# RAG (Retrieval-Augmented Generation) dengan LangChain & LangGraph

## Apa itu RAG?

LLM seperti Llama atau Gemini punya pengetahuan yang terbatas pada data training-nya. Mereka tidak tahu:
- Dokumen internal perusahaan kamu
- Data terbaru setelah tanggal cutoff
- Informasi spesifik yang tidak ada di internet

**RAG** adalah teknik untuk mengatasi ini. Alih-alih mengandalkan memori model, kita ambil informasi yang relevan dari database dulu, lalu kirimkan ke model sebagai konteks.

### Arsitektur RAG

```
Dokumen PDF/Text
      ↓
   Chunking          ← potong jadi bagian kecil
      ↓
   Embedding         ← ubah teks jadi vector angka
      ↓
  Vector Store       ← simpan di FAISS
      ↓
User tanya sesuatu
      ↓
  Retrieve           ← cari chunk yang paling relevan
      ↓
  Generate           ← kirim chunk + pertanyaan ke LLM
      ↓
   Jawaban
```

### Yang akan kita bangun di notebook ini:

1. **Vector Database** dengan FAISS — simpan dan cari dokumen berdasarkan kemiripan semantik
2. **RAG Pipeline** dengan LangGraph — alur retrieve → generate yang terstruktur

## Setup & Instalasi

Install dependencies:
```bash
pip install -r requirements.txt
```

Buat file `.env` di folder ini:
```
GOOGLE_API_KEY=your_gemini_api_key
GROQ_API_KEY=your_groq_api_key
```

Package yang dibutuhkan:
- `langchain`, `langchain-community`, `langchain-text-splitters` — framework utama
- `langchain-google-genai` — untuk Google Embedding
- `langchain-groq` — untuk LLM Llama via Groq
- `faiss-cpu` — vector database
- `PyMuPDF` — baca file PDF
- `langgraph` — orkestrasi alur RAG
- `python-dotenv` — load API key dari file `.env`

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # membaca GOOGLE_API_KEY dan GROQ_API_KEY dari file .env

# Part 1: Vector Database dengan FAISS

## Apa itu Embedding?

Embedding adalah proses mengubah teks menjadi deretan angka (vector) yang merepresentasikan makna teks tersebut.

```
"Saya suka makan nasi goreng"  →  [0.12, -0.34, 0.87, 0.03, ...]  (768 angka)
"I love fried rice"            →  [0.11, -0.31, 0.85, 0.02, ...]  (mirip!)
"Harga saham naik hari ini"    →  [-0.55, 0.22, -0.14, 0.91, ...]  (jauh berbeda)
```

Teks yang maknanya mirip akan menghasilkan vector yang "dekat" satu sama lain di ruang vektor.
Inilah yang memungkinkan pencarian semantik — kita tidak mencari keyword, tapi mencari makna.

## Apa itu FAISS?

**FAISS** (Facebook AI Similarity Search) adalah library untuk menyimpan dan mencari vector secara efisien.
Kita bisa memasukkan ribuan dokumen, lalu mencari dokumen yang paling mirip dengan query dalam hitungan milidetik.

## Inisialisasi Embedding & FAISS

Kita pakai `gemini-embedding-2` dari Google sebagai model embedding.
Model ini mengubah setiap teks menjadi vector 768 dimensi.

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# text-embedding-004 adalah model embedding terbaru dari Google
# menghasilkan vector 768 dimensi per teks
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2",
    output_dimensionality=768
)

In [ ]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

# embed_query("hello world") menghasilkan vector — kita ambil panjangnya
# untuk menentukan dimensi index FAISS
# IndexFlatL2 = cari vector terdekat dengan jarak Euclidean (L2)
index = faiss.IndexFlatL2(len(embeddings.embed_query("hello world")))

# Inisialisasi FAISS Vector Store
vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),  # simpan dokumen di memori (bukan disk)
    index_to_docstore_id={},      # mapping index FAISS ke ID dokumen
)

## Tambah Dokumen ke Vector Store

Setiap dokumen direpresentasikan sebagai objek `Document` yang punya dua field:
- `page_content` — isi teks dokumen
- `metadata` — informasi tambahan (bisa dipakai untuk filtering)

In [ ]:
from uuid import uuid4
from langchain_core.documents import Document

document_1 = Document(page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.", metadata={"source": "tweet"})
document_2 = Document(page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.", metadata={"source": "news"})
document_3 = Document(page_content="Building an exciting new project with LangChain - come check it out!", metadata={"source": "tweet"})
document_4 = Document(page_content="Robbers broke into the city bank and stole $1 million in cash.", metadata={"source": "news"})
document_5 = Document(page_content="Wow! That was an amazing movie. I can't wait to see it again.", metadata={"source": "tweet"})
document_6 = Document(page_content="Is the new iPhone worth the price? Read this review to find out.", metadata={"source": "website"})
document_7 = Document(page_content="The top 10 soccer players in the world right now.", metadata={"source": "website"})
document_8 = Document(page_content="LangGraph is the best framework for building stateful, agentic applications!", metadata={"source": "tweet"})
document_9 = Document(page_content="The stock market is down 500 points today due to fears of a recession.", metadata={"source": "news"})
document_10 = Document(page_content="I have a bad feeling I am going to get deleted :(", metadata={"source": "tweet"})

documents = [document_1, document_2, document_3, document_4, document_5,
             document_6, document_7, document_8, document_9, document_10]

# UUID dipakai sebagai ID unik untuk setiap dokumen
uuids = [str(uuid4()) for _ in range(len(documents))]

vector_store.add_documents(documents=documents, ids=uuids)

## Pencarian di FAISS

FAISS mendukung dua jenis pencarian:

**1. Similarity Search** — cari dokumen paling mirip dengan query secara semantik

**2. Similarity Search dengan Filter** — sama seperti di atas, tapi disaring berdasarkan metadata

| Operator | Arti |
|---|---|
| `$eq` | sama dengan |
| `$neq` | tidak sama dengan |
| `$gt` / `$lt` | lebih besar / lebih kecil |
| `$gte` / `$lte` | lebih besar atau sama / lebih kecil atau sama |
| `$in` | ada di dalam list |
| `$nin` | tidak ada di dalam list |
| `$and` | semua kondisi harus terpenuhi |
| `$or` | salah satu kondisi terpenuhi |
| `$not` | negasi kondisi |

In [ ]:
# Cari 2 dokumen paling mirip dengan query, tapi hanya dari sumber 'tweet'
results = vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=2,
    filter={"source": {"$eq": "tweet"}},
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

In [ ]:
# Cari tanpa filter — dari semua sumber
results = vector_store.similarity_search("The new iPhone", k=2)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

## Simpan dan Muat FAISS Index

FAISS bisa disimpan ke disk agar tidak perlu embed ulang setiap kali program dijalankan.

In [ ]:
# Simpan index ke folder lokal
vector_store.save_local("faiss_index")

In [ ]:
# Muat kembali dari disk
# allow_dangerous_deserialization=True diperlukan karena FAISS menggunakan pickle
new_vector_store = FAISS.load_local(
    "faiss_index", embeddings, allow_dangerous_deserialization=True
)

docs = new_vector_store.similarity_search("qux")
for res in docs:
    print(f"* {res.page_content} [{res.metadata}]")

# Part 2: RAG Pipeline dengan LangChain & LangGraph

Sekarang kita bangun RAG yang sesungguhnya — membaca dokumen PDF, menyimpannya ke FAISS,
lalu menjawab pertanyaan berdasarkan isi dokumen tersebut.

## Mengapa LangGraph?

LangGraph memungkinkan kita mendefinisikan alur RAG sebagai **graph** dengan node dan edge yang jelas.
Ini lebih terstruktur dibanding chain biasa karena:
- Mudah divisualisasikan
- Mudah ditambah node baru (misal: reranking, self-reflection)
- State dikelola secara eksplisit — setiap node tahu apa yang sudah dikerjakan node sebelumnya

Alur yang akan kita buat:
```
START → retrieve → generate → END
```

In [ ]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document
import fitz  # PyMuPDF

### Langkah 1: Extract dan Chunk Dokumen

**Mengapa perlu chunking?**

LLM punya batas token — kita tidak bisa memasukkan seluruh isi dokumen 100 halaman sekaligus.
Selain itu, semakin panjang konteks yang dikirim, semakin lambat dan mahal responsnya.

Dengan chunking, dokumen dipotong jadi bagian-bagian kecil. Saat user bertanya, kita hanya ambil
chunk yang paling relevan — bukan seluruh dokumen.

**Parameter chunking:**
- `chunk_size=1000` — maksimal 1000 karakter per chunk
- `chunk_overlap=200` — 200 karakter terakhir chunk sebelumnya diulang di chunk berikutnya
- `separator="\n"` — potong di newline dulu sebelum memotong berdasarkan panjang

**Letakkan file PDF kamu di folder `docs/` lalu update variabel `PDF_PATH` di bawah.**

In [ ]:
def extract_text_from_pdf(pdf_path: str) -> str:
    """Mengekstrak teks dari PDF menggunakan PyMuPDF"""
    doc = fitz.open(pdf_path)
    return "\n".join([page.get_text() for page in doc])

# Ganti path ini dengan PDF kamu
PDF_PATH = "docs/Rizky_Andika_CV.pdf"

pdf_text = extract_text_from_pdf(PDF_PATH)
print(f"Total karakter: {len(pdf_text)}")
print("\nPreview 500 karakter pertama:")
print(pdf_text[:500])

In [ ]:
splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_text(pdf_text)
print(f"Total chunks: {len(chunks)}")

In [ ]:
# Preview setiap chunk (100 karakter pertama saja)
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk[:100]}...")

### Langkah 2: Upload Chunk ke Vector Store

Setelah dokumen di-chunk, kita embed dan simpan ke FAISS.
Proses ini hanya dilakukan sekali — hasilnya disimpan ke disk agar bisa dimuat ulang.

In [ ]:
# Inisialisasi FAISS baru untuk dokumen PDF ini
index = faiss.IndexFlatL2(len(embeddings.embed_query("hello world")))

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [ ]:
# Bungkus setiap chunk sebagai Document object
documents = [Document(page_content=chunk) for chunk in chunks]

uuids = [str(uuid4()) for _ in range(len(documents))]

# Embed dan simpan ke FAISS
vector_store.add_documents(documents=documents, ids=uuids)
print(f"{len(documents)} chunks berhasil disimpan ke vector store")

In [ ]:
# Simpan ke disk agar tidak perlu embed ulang
vector_store.save_local("faiss_index")
print("Index tersimpan di folder faiss_index/")

### Langkah 3: Bangun RAG dengan LangGraph

Kita definisikan tiga komponen utama:

**1. State** — objek yang membawa data antar node di graph. Berisi:
- `question` — pertanyaan dari user
- `context` — dokumen yang diambil dari FAISS
- `answer` — jawaban final dari LLM

**2. Node `retrieve`** — ambil chunk relevan dari FAISS berdasarkan pertanyaan

**3. Node `generate`** — kirim pertanyaan + chunk ke LLM untuk dijawab

In [ ]:
from langchain_core.documents import Document
from typing import List, TypedDict
from langchain_groq import ChatGroq

# State adalah "tas" yang dibawa sepanjang perjalanan graph
# Setiap node bisa membaca dan mengisi field di State
class State(TypedDict):
    question: str           # pertanyaan dari user
    context: List[Document] # hasil retrieve dari FAISS
    answer: str             # jawaban final dari LLM

#### RAG Prompt

Kita pakai prompt standar dari LangChain Hub (`rlm/rag-prompt`).
Prompt ini sudah dirancang khusus untuk RAG — menginstruksikan model untuk hanya menjawab
berdasarkan konteks yang diberikan, dan mengatakan "saya tidak tahu" jika informasinya tidak ada.

In [ ]:
from langchain import hub

# Tarik prompt RAG standar dari LangChain Hub
# Prompt ini punya dua variabel: {context} dan {question}
prompt = hub.pull("rlm/rag-prompt")

# Lihat isi prompt-nya
example_messages = prompt.invoke(
    {"context": "(context goes here)", "question": "(question goes here)"}
).to_messages()

print("Isi RAG prompt:")
print(example_messages[0].content)

### Cara Mendapatkan Groq API Key

**Groq** adalah platform inference yang menyediakan akses gratis ke model open-source seperti Llama, Mixtral, dan Gemma.

#### Langkah-langkah:

1. Buka [https://console.groq.com](https://console.groq.com) dan buat akun (bisa pakai Google)
2. Setelah login, klik menu **API Keys** di sidebar kiri
3. Klik **Create API Key**, beri nama, lalu copy key-nya
4. Tambahkan ke file `.env` di folder ini:
   ```
   GROQ_API_KEY=gsk_xxxxxxxxxxxx
   ```

> **Catatan**: Groq menyediakan free tier dengan rate limit yang cukup untuk eksperimen.
> Untuk melihat daftar model yang tersedia, cek [https://console.groq.com/docs/models](https://console.groq.com/docs/models)

In [ ]:
# Inisialisasi LLM — pakai Llama 3.3 70B via Groq (gratis, cepat)
MODEL = "llama-3.3-70b-versatile"

llm = ChatGroq(
    temperature=0,  # 0 = deterministik, cocok untuk Q&A faktual
    model=MODEL
)

In [ ]:
# Node 1: Retrieve — ambil dokumen relevan dari FAISS
def retrieve(state: State):
    retrieved_docs = vector_store.similarity_search(state["question"])
    return {"context": retrieved_docs}


# Node 2: Generate — buat jawaban berdasarkan context + question
def generate(state: State):
    # Gabungkan semua chunk yang diambil jadi satu string konteks
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    # Isi template prompt dengan konteks dan pertanyaan
    messages = prompt.invoke({"question": state["question"], "context": docs_content})
    # Kirim ke LLM
    response = llm.invoke(messages)
    return {"answer": response.content}

In [ ]:
from langgraph.graph import START, StateGraph

# Definisikan graph: retrieve dulu, lalu generate
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

print("Graph berhasil dikompilasi!")

In [ ]:
# Visualisasi alur graph
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

## Jalankan RAG

Sekarang kita bisa bertanya tentang isi dokumen PDF yang sudah kita upload.
LLM akan menjawab hanya berdasarkan konten dokumen — bukan dari pengetahuan umumnya.

In [ ]:
result = graph.invoke({"question": "Who is Rizky Andika?"})

print("=== Context yang diambil dari FAISS ===")
for i, doc in enumerate(result["context"]):
    print(f"\nChunk {i+1}:")
    print(doc.page_content[:200] + "...")

print("\n=== Jawaban LLM ===")
print(result["answer"])

In [ ]:
result = graph.invoke({"question": "When and where is Rizky Andika graduated?"})
print(f'Answer: {result["answer"]}')